# Simulation Plotting Tools

## Import Data

In [60]:
import numpy as np
import uproot
import pandas as pd
import random

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

In [68]:
filepath = "output/"

# Load Kalman and actual hit data for each track
with uproot.open(filepath+"kalman_output.root") as file:
    tree = file["Particles"]
    df = tree.arrays(["KalmanFitted/KalmanFitted.hitPos.fCoordinates.fX", \
                      "KalmanFitted/KalmanFitted.hitPos.fCoordinates.fY", \
                      "KalmanFitted/KalmanFitted.hitPos.fCoordinates.fZ", \
                      
                      "Actual/Actual.entryPos.fCoordinates.fX", \
                      "Actual/Actual.entryPos.fCoordinates.fY", \
                      "Actual/Actual.entryPos.fCoordinates.fZ",\
                      
                      "KalmanFitted/KalmanFitted.hitMom.fCoordinates.fX",\
                      "KalmanFitted/KalmanFitted.hitMom.fCoordinates.fY",\
                      "KalmanFitted/KalmanFitted.hitMom.fCoordinates.fZ",\

                      "Actual/Actual.initMom.fCoordinates.fX",\
                      "Actual/Actual.initMom.fCoordinates.fY",\
                      "Actual/Actual.initMom.fCoordinates.fZ",\
                      
                      "KalmanFitted/KalmanFitted.trackID",\
                      "Actual/Actual.trackID"], library="pd")

# There is only one row of the data
row = df.iloc[0]

# Load layer configuration and sample event row data
layer_df = pd.read_csv(filepath+'layer_radius.csv')

# Load cluster data
clusters_df = pd.read_csv(filepath+'cluster_info.csv')
clusters_df["hit_time"] = clusters_df["gen_time"] + clusters_df["drift_time"]

## Trajectory Plotting

In [40]:
# Plot the actual trajectory vs the kalman reconstructed trajectory
def plot_fit(track, side_view, add_wires):
    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    kalman_data = [[], [], []]
    actual_data = [[], [], []]
    
    # Extract Kalman and actual hit positions for the specified track
    for idx in range(len(row['KalmanFitted/KalmanFitted.trackID'])):
        if row['KalmanFitted/KalmanFitted.trackID'][idx] == track:
            kalman_data[0].append(row["KalmanFitted/KalmanFitted.hitPos.fCoordinates.fX"][idx])
            kalman_data[1].append(row["KalmanFitted/KalmanFitted.hitPos.fCoordinates.fY"][idx])
            kalman_data[2].append(row["KalmanFitted/KalmanFitted.hitPos.fCoordinates.fZ"][idx])
            
        try:
            if row['Actual/Actual.trackID'][idx] == track:
                actual_data[0].append(row["Actual/Actual.entryPos.fCoordinates.fX"][idx])
                actual_data[1].append(row["Actual/Actual.entryPos.fCoordinates.fY"][idx])
                actual_data[2].append(row["Actual/Actual.entryPos.fCoordinates.fZ"][idx])
        except (KeyError, IndexError):
            continue

    if add_wires:
        for layer_idx in range(len(layer_df)):
            
            # Get the wire layer geometry
            cur_df = layer_df.iloc[layer_idx]
            r = 0.5 * (cur_df['r1'] + cur_df['r2'])
            w_type = cur_df["type"]
            num_wires = cur_df["numWires"]
            wire_length = cur_df["wireLength"]
            delta = 2 * np.pi / num_wires

            # Alternate layers are staggered by half a cell
            offset = 0.0 if cur_df["i"] % 2 == 0 else delta / 2.0
            
            for wire_i in range(num_wires):
                t1 = offset + (wire_i + 0.5) * delta
                if w_type == "stereo+":
                    t2 = offset + (wire_i + 3.5) * delta # Skew 3 wire places forward
                elif w_type == "stereo-":
                    t2 = offset + (wire_i - 2.5) * delta # Skew 3 wire places backward
                else:
                    t2 = t1 # Axial wire so straight across

                # Get the start and end positions of the wire
                x = [r * np.cos(t1), r * np.cos(t2)]
                y = [r * np.sin(t1), r * np.sin(t2)]
                z = [-wire_length, wire_length]

                # Plot the sense wires
                if "stereo" in w_type:
                    ax.plot(x, y, z, color="tab:green", lw=0.5)
                else:
                    ax.plot(x, y, z, color="tab:blue", lw=0.5)

    # Plot fitted and actual tracks
    ax.plot(kalman_data[0], kalman_data[1], kalman_data[2], color="red", label="Kalman fitted")
    ax.plot(actual_data[0], actual_data[1], actual_data[2], color="black", label="Actual")

    # Make plot a 2D view if specified
    if side_view:
        ax.view_init(elev=90, azim=-90)
        
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")

    handles, labels = ax.get_legend_handles_labels()

    # Label the sense wires if added
    if add_wires:
        blue_line = Line2D([0], [0], color='tab:blue', label="Axial wires")
        green_line = Line2D([0], [0], color='tab:green', label="Stereo wires")
        handles.extend([blue_line, green_line])
    
    ax.legend(handles=handles)
    plt.show()

In [50]:
#plot_fit(2, False, False) # Plots 3D view 
#plot_fit(2, True, False)  # Plots 2D view
#plot_fit(2, True, True)  # Plots 2D view with sense wires

## Signal plotting

In [52]:
def plot_signal(m, G, tau):
    # Detector resolution jitter from diffusion values
    quad_sigma = np.std(np.sqrt(clusters_df["long_diffusion"]**2 + clusters_df["trans_diffusion"]**2), ddof=1)
    
    wire_times = []
    gain_list = []
    
    for time in clusters_df["hit_time"]:
        sampled_gain = np.random.gamma(shape=m, scale=G/m)
        gain_list.append(sampled_gain)
        
        quad_ran = random.gauss(0, quad_sigma)
        response_delay = np.random.gamma(shape=2, scale=tau)
        smeared_time = time + response_delay + quad_ran
        wire_times.append(smeared_time)

    # Order the data by increasing time
    sorted_pairs = sorted(zip(wire_times, gain_list))
    wire_times_sorted, gain_sorted = map(list, zip(*sorted_pairs))

    # Make all y values positive
    flipped_gain = abs((np.array(gain_sorted)-G))

    # Insert a value between each value in a list
    def insert_midpoints(lst):
        result = []
        for i in range(len(lst) - 1):
            result.append(lst[i])
            result.append((lst[i] + lst[i+1]) / 2)
        result.append(lst[-1])
        return result

    # Add a value with 0 gain after every point for clarity
    zeroed_times = insert_midpoints(wire_times_sorted[:150])
    zeroed_gains = [x for item in flipped_gain[:150] for x in (item, 0)]

    # Plot the signal function
    plt.figure(figsize=(12, 10))
    plt.plot(zeroed_times[:150], zeroed_gains[:150])
    plt.xlabel("Time [ns]")
    plt.ylabel("Charge/Gain")
    plt.show()

In [64]:
m = 10
G = 20000
tau = 8

#plot_signal(m, G, tau)